# Rank Followup
Configuration: `stages/02_rank_followup/config`. Figures remain local; datasets are under `experiment/data/`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
rank_followup_experiment_dir = ROOT / 'experiment/research/osd/stages/02_rank_followup'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

rank_followup_data_paths = ExperimentPaths(ROOT, rank_followup_experiment_dir / 'config/paths.json')


In [ ]:
from dataclasses import dataclass
from hashlib import sha256
from itertools import product
from math import comb
from pathlib import Path
from time import perf_counter
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import kendalltau, pearsonr, spearmanr
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from decoders.cpp_backend import Tesseract
from decoders.cython import BPOSD
from decoders.cython.cython_core.osd_core import ordered_statistics_postprocess
from noises_ca import CircuitBuilder, ErrorRate

rank_followup_config = json.loads((rank_followup_experiment_dir / 'config/reliability_rank_followup.json').read_text(encoding='utf-8'))
if rank_followup_config['smoke_mode']:
    rank_followup_config.update(
        levels=[2],
        event_p_values=[0.05],
        event_gap_targets_by_level={'2': 2},
        event_batch_size=40,
        event_max_shots_per_point=80,
        unbiased_p_values=[0.05],
        unbiased_shots_per_point=12,
        bp_iterations=8,
        snapshot_iterations=[1, 2, 4, 8],
        temporal_windows=[3, 5],
        instability_beta_values=[0.5],
        sign_gamma_values=[1.0],
        conflict_beta_values=[0.5],
        rank_threshold_values=[0.9],
        rank_persistence_values=[2],
        message_residual_quantiles=[0.5],
        combined_weight_values=[0.0, 1.0],
        bootstrap_repetitions=50,
        scaling_values=[1.0],
        damping_values=[0.0],
        schedule_values=['parallel'],
        iteration_values=[4, 8],
        hyperparameter_case_limit=8,
    )

# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# reuse_existing_data: boolean (True or False).
# levels: non-empty.
# levels: values restricted to {2, 3}.
# primary_level: chosen from levels.
# instance_weights_by_level: {'2': [2, 3], '3': [4, 5]}.
# event_p_values, unbiased_p_values: each item is in (0, 1).
# event_batch_size: positive.
# event_max_shots_per_point: positive.
# unbiased_shots_per_point: positive.
# Expected relationship: bp_iterations >= maximum of snapshot_iterations.
# osd_orders_by_level: include a string key for every entry in levels.
# values of osd_orders_by_level: each order list is sorted, unique, and begins with [0, 1].
# temporal_windows, top_k_values: each item is positive.
# snapshot_stability_window: positive.
# damping_values: each item is in [0, 1).
# scaling_values: each item is positive.
# schedule_values: values restricted to {'parallel', 'serial'}.
# development_fraction: in (0, 1).
# bootstrap_repetitions: positive.
# message_zero_tolerance: non-negative.

from experiment.utils.bp_trace import check_bp_cache
rank_followup_data_paths.run(smoke_mode=rank_followup_config['smoke_mode'])
check_bp_cache(rank_followup_data_paths, rank_followup_config)



In [ ]:
from dataclasses import dataclass
from hashlib import sha256
from itertools import product
from math import comb
from pathlib import Path
from time import perf_counter
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import kendalltau, pearsonr, spearmanr
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from decoders.cpp_backend import Tesseract
from decoders.cython import BPOSD
from decoders.cython.cython_core.osd_core import ordered_statistics_postprocess
from noises_ca import CircuitBuilder, ErrorRate
from experiment.utils import misc

rank_followup_read_csv = misc.read_csv

from experiment.utils import misc

rank_followup_bits = misc.bits

from experiment.utils import misc

rank_followup_bit_array = misc.bit_array_variant_3

from experiment.utils import osd_reliability

rank_followup_Context = osd_reliability.Context_variant_2

from experiment.utils import osd_reliability

rank_followup_BPTrace = osd_reliability.BPTrace
rank_followup_context_cache = {}

from experiment.utils import osd_reliability

rank_followup_gf2_rank = osd_reliability.gf2_rank_variant_2

from experiment.utils import osd_reliability

def rank_followup_context_for(level, p):
    return osd_reliability.context_for(
        level,
        p,
        CA_9_2_5=CA_9_2_5,
        CONTEXT_CACHE=rank_followup_context_cache,
        CircuitBuilder=CircuitBuilder,
        Context=rank_followup_Context,
        ErrorRate=ErrorRate,
        TriangularCode=TriangularCode,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
        gf2_rank=rank_followup_gf2_rank,
    )

from experiment.utils import osd_reliability

def rank_followup_trace_min_sum(
    context,
    syndrome,
    iterations,
    scaling=1.0,
    damping=0.0,
    schedule='parallel',
    early_stop=False,
):
    return osd_reliability.trace_min_sum(context, syndrome, iterations, scaling, damping, schedule, early_stop, BPTrace=rank_followup_BPTrace)

from experiment.utils import osd_reliability

rank_followup_logical_label = osd_reliability.logical_label

from experiment.utils import osd_reliability

def rank_followup_outcome(
    context,
    correction,
    syndrome,
    truth,
    error=None,
):
    return osd_reliability.outcome(context, correction, syndrome, truth, error, logical_label=rank_followup_logical_label)

from experiment.utils import osd_reliability

rank_followup_ordering_proxy = osd_reliability.ordering_proxy

from experiment.utils import osd_reliability

def rank_followup_osd_from_reliability(
    context,
    syndrome,
    reliability,
    order,
):
    return osd_reliability.osd_from_reliability(context, syndrome, reliability, order, ordered_statistics_postprocess=ordered_statistics_postprocess, ordering_proxy=rank_followup_ordering_proxy)

from experiment.utils import osd_reliability

rank_followup_ordinal_ranks = osd_reliability.ordinal_ranks

from experiment.utils import osd_reliability

rank_followup_safe_corr = osd_reliability.safe_corr
rank_followup_trace_cache = {}


In [ ]:
_input_run = 'main'
rank_followup_event_screen = rank_followup_read_csv(rank_followup_data_paths.input('event_screen_csv', _input_run=_input_run))
rank_followup_unbiased_cases = rank_followup_read_csv(rank_followup_data_paths.input('unbiased_cases_csv', _input_run=_input_run))


# Follow-up: instability-guided reliability-rank experiments

  All BP-internal
quantities and every BP/OSD decision below are produced by `trace_min_sum` and
`osd_from_reliability`; the Cython BPOSD wrapper used by earlier sections is not
called here.  The primary mechanism cohort is d=5, weight 2/3.  d=9 uses only
weight 4/5 and is reported as exploratory unless enough gap cases are present.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

FOLLOWUP_CONFIG = json.loads((rank_followup_experiment_dir / 'config/followup_config.json').read_text(encoding='utf-8'))
if rank_followup_config['smoke_mode']:
    FOLLOWUP_CONFIG.update(
        screen_trace_limits_by_distance={'5': 40, '9': 20},
        case_limits_by_distance={'5': 12, '9': 6},
        minimum_gap_events=2,
        top_k_values=[3, 5],
        cv_folds=2,
        rank_weight_values=[0.0, 1.0],
        pair_limit_per_shot=20,
        scaling_values=[1.0],
        damping_values=[0.0],
        trigger_quantiles=[0.5, 0.8],
        bootstrap_repetitions=50,
    )

# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# reuse_existing_data: boolean (True or False).
# schema_version: positive integer.
# primary_distance: 5.
# development_fraction: in (0, 1).
# top_k_values: each item is positive.
# damping_values: each item is in [0, 1).
# scaling_values: each item is positive.
# trigger_quantiles: each item is in (0, 1).
# cv_folds: at least 2.
# screen_trace_limits_by_distance: keys exactly {'5', '9'}.
# case_limits_by_distance: keys exactly {'5', '9'}.
# values of screen_trace_limits_by_distance: each item is positive.
# values of case_limits_by_distance: each item is positive.
# osd_orders_by_distance: keys exactly {'5', '9'}.
# values of osd_orders_by_distance: each order list is sorted, unique, and begins with [0, 1].
followup_name = 'reliability_rank_followup_smoke' if rank_followup_config['smoke_mode'] else 'reliability_rank_followup'
FOLLOWUP_OUT = rank_followup_data_paths.run(smoke_mode=rank_followup_config['smoke_mode'])

from decoders.cython.cython_core.osd_core import OSD_VERSION, osd_candidate_count
from experiment.utils.misc import check_algorithm_cache
check_algorithm_cache(rank_followup_data_paths, FOLLOWUP_CONFIG, {'osd': OSD_VERSION})

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(rank_followup_data_paths, FOLLOWUP_CONFIG)

FOLLOWUP_STATS = rank_followup_data_paths.output('stats')
FOLLOWUP_TRACE = rank_followup_data_paths.output('trace')
FOLLOWUP_CONF = rank_followup_data_paths.output('config')

for rank_followup_directory in (FOLLOWUP_STATS, FOLLOWUP_TRACE, FOLLOWUP_CONF):
    rank_followup_directory.mkdir(parents=True, exist_ok=True)

followup_config_path = rank_followup_data_paths.output('config_json')
if not followup_config_path.is_file() or not FOLLOWUP_CONFIG['reuse_existing_data']:
    followup_config_path.write_text(json.dumps({'config': FOLLOWUP_CONFIG, 'base_config': rank_followup_config}, indent=2), encoding='utf-8')

def followup_log(level, payload):
    if FOLLOWUP_CONFIG['print_level'] >= level:
        print(payload)

def followup_stage_cached(paths):
    present = [path.is_file() for path in paths]
    assert all(present) or not any(present), f'partial follow-up cache: {[str(p) for p, ok in zip(paths, present) if ok]}'
    return FOLLOWUP_CONFIG['reuse_existing_data'] and all(present)

def followup_split(shot_id):
    digest = sha256(f"{FOLLOWUP_CONFIG['random_seed']}:{shot_id}".encode()).digest()
    value = int.from_bytes(digest[:8], 'little') / 2 ** 64
    return 'development' if value < FOLLOWUP_CONFIG['development_fraction'] else 'test'

def followup_bootstrap(values, seed_offset=0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not len(values):
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(FOLLOWUP_CONFIG['random_seed'] + seed_offset)
    means = np.asarray([rng.choice(values, len(values), replace=True).mean() for _ in range(FOLLOWUP_CONFIG['bootstrap_repetitions'])])

    return (float(values.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975)))

def score_from_order(order):
    order = np.asarray(order, dtype=int)
    assert np.array_equal(np.sort(order), np.arange(len(order)))
    score = np.empty(len(order), dtype=float)
    score[order] = np.arange(len(order), dtype=float)

    return score

def minimal_topk_promotion(order, repair_mask, k):
    order = np.asarray(order, dtype=int)
    repair_mask = np.asarray(repair_mask, dtype=bool)
    k = min(int(k), len(order))
    repair_ids = set(np.flatnonzero(repair_mask))
    assert len(repair_ids) <= k
    original_top = list(map(int, order[:k]))
    promoted = [int(variable) for variable in order[k:] if variable in repair_ids]
    displaced = [variable for variable in reversed(original_top) if variable not in repair_ids][:len(promoted)]
    new_top_set = set(original_top) - set(displaced) | set(promoted)
    new_top = [int(variable) for variable in order if variable in new_top_set]
    new_tail = [int(variable) for variable in order if variable not in new_top_set]
    result = np.asarray(new_top + new_tail)
    assert len(new_top) == k and repair_ids <= set(new_top)
    assert len(displaced) == len(promoted)

    return result

def first_valid_or_final(trace):
    return int(trace.first_valid) if trace.first_valid is not None else len(trace.llr) - 1

def traced_decode_bundle(
    context,
    syndrome,
    truth,
    error,
    trace=None,
    scaling=1.0,
    damping=0.0,
):
    trace = rank_followup_trace_min_sum(context, syndrome, rank_followup_config['bp_iterations'], scaling=scaling, damping=damping, schedule='parallel') if trace is None else trace
    endpoint = first_valid_or_final(trace)
    bp = trace.hard[endpoint]
    full_order = FOLLOWUP_CONFIG['osd_orders_by_distance'][str(context.distance)][-1]
    standard = bp if trace.valid[endpoint] else rank_followup_osd_from_reliability(context, syndrome, trace.llr[endpoint], full_order)
    forced = rank_followup_osd_from_reliability(context, syndrome, trace.llr[endpoint], full_order)

    return {
        'trace': trace,
        'endpoint': endpoint,
        'bp': bp,
        'standard': standard,
        'forced': forced,
        'bp_outcome': rank_followup_outcome(context, bp, syndrome, truth, error),
        'standard_outcome': rank_followup_outcome(context, standard, syndrome, truth, error),
        'forced_outcome': rank_followup_outcome(context, forced, syndrome, truth, error),
    }

FOLLOWUP_TRACE_CACHE = {}

def followup_case_material(case, scaling=1.0, damping=0.0):
    context = rank_followup_context_for(int(case.level), float(case.p))
    syndrome = rank_followup_bit_array(case.syndrome, context.H.shape[0])
    truth = rank_followup_bit_array(case.true_logical, context.L.shape[0])
    error = rank_followup_bit_array(case.error, context.H.shape[1])
    key = (str(case.shot_id), float(scaling), float(damping))

    if key not in FOLLOWUP_TRACE_CACHE:
        FOLLOWUP_TRACE_CACHE[key] = rank_followup_trace_min_sum(context, syndrome, rank_followup_config['bp_iterations'], scaling=scaling, damping=damping, schedule='parallel')
    return (context, syndrome, truth, error, FOLLOWUP_TRACE_CACHE[key])

def prefix_features(context, trace, endpoint):
    endpoint = int(endpoint)
    assert 0 <= endpoint < len(trace.llr)
    if endpoint == 0:
        zeros = np.zeros(context.H.shape[1], dtype=float)
        current_abs = np.abs(trace.llr[0])
        return {
            'current_abs_llr': current_abs,
            'message_instability': zeros.copy(),
            'belief_instability': zeros.copy(),
            'sign_flip': zeros.copy(),
            'extrinsic_conflict': zeros.copy(),
            'rank_volatility': zeros.copy(),
            'sign_consistency': np.abs(np.sign(trace.llr[0])),
            'llr_variance': zeros.copy(),
            'ranks': rank_followup_ordinal_ranks(current_abs),
        }

    start = max(0, endpoint - FOLLOWUP_CONFIG['snapshot_feature_window'] + 1)
    recent = trace.llr[start:endpoint + 1]
    current_abs = np.abs(trace.llr[endpoint])
    rank_rows = np.asarray([rank_followup_ordinal_ranks(np.abs(row)) for row in trace.llr[:endpoint + 1]])
    rank_volatility = np.abs(np.diff(rank_rows, axis=0)).mean(axis=0)
    belief_instability = np.abs(np.diff(trace.llr[:endpoint + 1], axis=0)).mean(axis=0)
    belief_instability /= np.abs(context.prior_llr) + np.finfo(float).eps
    c2v = trace.c2v[1:endpoint + 1]
    delta = np.abs(np.diff(trace.c2v[:endpoint + 1], axis=0))
    delta /= np.abs(context.prior_llr[context.edge_variables])[None, :] + np.finfo(float).eps
    message_instability = np.zeros(context.H.shape[1])
    sign_flip = np.zeros(context.H.shape[1])
    conflict = np.zeros(context.H.shape[1])
    eps = np.finfo(float).eps

    for variable, ids in enumerate(context.variable_edges):
        message_instability[variable] = delta[:, ids].mean()
        signs = np.sign(c2v[:, ids])
        comparable = (signs[1:] != 0) & (signs[:-1] != 0)
        sign_flip[variable] = np.mean((signs[1:] != signs[:-1]) & comparable) if len(signs) > 1 else 0.0
        incoming = c2v[:, ids]
        conflict[variable] = np.mean(1 - np.abs(incoming.sum(axis=1)) / (np.abs(incoming).sum(axis=1) + eps))

    return {
        'current_abs_llr': current_abs,
        'message_instability': message_instability,
        'belief_instability': belief_instability,
        'sign_flip': sign_flip,
        'extrinsic_conflict': conflict,
        'rank_volatility': rank_volatility,
        'sign_consistency': np.abs(np.sign(recent).mean(axis=0)),
        'llr_variance': recent.var(axis=0),
        'ranks': rank_rows[-1],
    }

followup_log(2, {'output': str(FOLLOWUP_OUT)})


## Trace-native cohort reconstruction

`standard_trace` accepts the first syndrome-valid BP hard decision; if no BP
iterate is valid, it applies the configured full OSD order to the final traced
reliability. `forced_trace` applies the same OSD postprocessor regardless of BP
validity.  Group B is therefore defined without consulting Cython labels.

Reconstruct the cohorts using the trace-native baseline definitions before evaluating ranks. This prevents a different stopping or fallback convention from being interpreted as a ranking improvement.


In [ ]:
classification_path = rank_followup_data_paths.output('followup_case_classification_csv')
cohort_path = rank_followup_data_paths.output('followup_cohort_csv')
if followup_stage_cached([classification_path, cohort_path]):
    followup_classification = rank_followup_read_csv(classification_path)
    followup_cohort = rank_followup_read_csv(cohort_path)
else:
    selected_screen_parts = []
    for rank_followup_distance, rank_followup_frame in rank_followup_event_screen.groupby('distance', sort=True):
        expected = {2, 3} if int(rank_followup_distance) == 5 else {4, 5}
        assert set(rank_followup_frame.error_weight.unique()) <= expected
        limit = FOLLOWUP_CONFIG['screen_trace_limits_by_distance'][str(int(rank_followup_distance))]
        proposals = pd.concat([rank_followup_frame[rank_followup_frame.primary_group == 'B'].sort_values('shot_id'), rank_followup_frame[rank_followup_frame.primary_group != 'B'].sort_values('shot_id')]).drop_duplicates('shot_id')
        selected_screen_parts.append(proposals.head(limit))

    selected_screen = pd.concat(selected_screen_parts, ignore_index=True)
    rank_followup_rows = []
    for rank_followup_case in tqdm(selected_screen.itertuples(index=False), total=len(selected_screen), desc='trace-native cohort classification', disable=FOLLOWUP_CONFIG['print_level'] == 1):
        rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
        bundle = traced_decode_bundle(rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace)
        rank_followup_standard_correct = bundle['standard_outcome'][1]
        rank_followup_forced_correct = bundle['forced_outcome'][1]
        primary_group = 'A' if rank_followup_standard_correct and rank_followup_forced_correct else 'B' if not rank_followup_standard_correct and rank_followup_forced_correct else 'C' if not rank_followup_standard_correct and (not rank_followup_forced_correct) else 'other'
        rank_followup_repair = bundle['bp'] ^ bundle['forced']
        rank_followup_rows.append({
            'shot_id': rank_followup_case.shot_id,
            'level': int(rank_followup_case.level),
            'distance': rank_followup_context.distance,
            'p': float(rank_followup_case.p),
            'split': followup_split(rank_followup_case.shot_id),
            'error_weight': int(rank_followup_error.sum()),
            'syndrome_weight': int(rank_followup_syndrome.sum()),
            'syndrome': rank_followup_case.syndrome,
            'true_logical': rank_followup_case.true_logical,
            'error': rank_followup_case.error,
            'tesseract_correct': int(rank_followup_case.tesseract_correct),
            'primary_group': primary_group,
            'group_D': int(not bundle['bp_outcome'][0]),
            'bp_syndrome_consistent': bundle['bp_outcome'][0],
            'bp_logical_correct': bundle['bp_outcome'][1],
            'standard_trace_correct': rank_followup_standard_correct,
            'forced_trace_correct': rank_followup_forced_correct,
            'endpoint_iteration': bundle['endpoint'],
            'first_valid_iteration': np.nan if rank_followup_trace.first_valid is None else rank_followup_trace.first_valid,
            'bp_correction': rank_followup_bits(bundle['bp']),
            'forced_correction': rank_followup_bits(bundle['forced']),
            'repair_support': rank_followup_bits(rank_followup_repair),
            'repair_support_size': int(rank_followup_repair.sum()),
        })

    followup_classification = pd.DataFrame(rank_followup_rows)
    rank_followup_selected_rows = []
    for rank_followup_distance, rank_followup_frame in followup_classification.groupby('distance', sort=True):
        limit = FOLLOWUP_CONFIG['case_limits_by_distance'][str(int(rank_followup_distance))]
        gap_mask = rank_followup_frame.primary_group == 'B'
        if FOLLOWUP_CONFIG['strict_tesseract_gap']:
            strict = gap_mask & (rank_followup_frame.tesseract_correct == 1)
            gap_mask = strict if strict.any() else gap_mask
        groups = {
            'B': strict if FOLLOWUP_CONFIG['strict_tesseract_gap'] else gap_mask,
            'A': rank_followup_frame.primary_group == 'A',
            'C': rank_followup_frame.primary_group == 'C',
            'D': rank_followup_frame.group_D == 1,
        }

        for rank_followup_name, rank_followup_mask in groups.items():
            part = rank_followup_frame[rank_followup_mask].sort_values(['p', 'error_weight', 'shot_id']).head(limit).copy()
            part['cohort_group'] = rank_followup_name
            rank_followup_selected_rows.append(part)

    followup_cohort = pd.concat(rank_followup_selected_rows, ignore_index=True) if rank_followup_selected_rows else pd.DataFrame()
    assert len(followup_cohort), 'trace-native cohort is empty'
    followup_classification.to_csv(classification_path, index=False)
    followup_cohort.to_csv(cohort_path, index=False)

status_source = followup_classification.copy()
status_source['eligible_gap'] = (status_source.primary_group == 'B').astype(int)
if FOLLOWUP_CONFIG['strict_tesseract_gap']:
    status_source['eligible_gap'] *= status_source.tesseract_correct.astype(int)
followup_status = status_source.groupby('distance', as_index=False).agg(screened_cases=('shot_id', 'size'), gap_cases=('eligible_gap', 'sum'))
followup_status['mechanism_status'] = np.where((followup_status.distance == FOLLOWUP_CONFIG['primary_distance']) | (followup_status.gap_cases >= FOLLOWUP_CONFIG['minimum_gap_events']), 'enabled', 'exploratory_insufficient_gap_events')

followup_status.to_csv(rank_followup_data_paths.output('experiment_status_csv'), index=False)
followup_status


## Experiments 1--2: repair Top-K coverage and mis-ranking predictors

Measure where the variables changed by a successful OSD repair lie in the reliability ordering. Compare trajectory features as predictors of mis-ranking; the repair labels themselves are offline information.


In [ ]:
repair_trajectory_path = rank_followup_data_paths.output('repair_rank_trajectories_csv')
misranked_path = rank_followup_data_paths.output('misranked_variables_csv')
if followup_stage_cached([repair_trajectory_path, misranked_path]):
    repair_rank_trajectories = rank_followup_read_csv(repair_trajectory_path)
    misranked_variables = rank_followup_read_csv(misranked_path)
else:
    repair_rows, rank_followup_variable_rows = ([], [])
    gap_cases = followup_cohort[followup_cohort.cohort_group == 'B'].drop_duplicates('shot_id')
    for rank_followup_case in tqdm(gap_cases.itertuples(index=False), total=len(gap_cases), desc='repair-rank trajectories', disable=FOLLOWUP_CONFIG['print_level'] == 1):
        rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
        rank_followup_repair = rank_followup_bit_array(rank_followup_case.repair_support, rank_followup_context.H.shape[1]).astype(bool)
        assert rank_followup_repair.any()
        final_ranks = rank_followup_ordinal_ranks(np.abs(rank_followup_trace.llr[-1]))

        for rank_followup_iteration in range(1, len(rank_followup_trace.llr)):
            features = prefix_features(rank_followup_context, rank_followup_trace, rank_followup_iteration)
            rank_followup_ranks = features['ranks']
            for k_requested in FOLLOWUP_CONFIG['top_k_values']:
                rank_followup_k = min(k_requested, len(rank_followup_ranks))
                unreliable = rank_followup_ranks <= rank_followup_k
                coverage = float(np.mean(unreliable[rank_followup_repair]))

                for rank_followup_order in [0, 1]:
                    rank_followup_correction = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_trace.llr[rank_followup_iteration], rank_followup_order)
                    rank_followup_correct = rank_followup_outcome(rank_followup_context, rank_followup_correction, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1]
                    repair_rows.append({
                        'shot_id': rank_followup_case.shot_id,
                        'distance': rank_followup_context.distance,
                        'p': rank_followup_context.p,
                        'split': rank_followup_case.split,
                        'iteration': rank_followup_iteration,
                        'relative_iteration': np.nan if pd.isna(rank_followup_case.first_valid_iteration) else rank_followup_iteration - int(rank_followup_case.first_valid_iteration),
                        'K_requested': k_requested,
                        'K': rank_followup_k,
                        'Q_K': coverage,
                        'mean_repair_rank': float(rank_followup_ranks[rank_followup_repair].mean()),
                        'max_repair_rank': int(rank_followup_ranks[rank_followup_repair].max()),
                        'osd_order': rank_followup_order,
                        'logical_correct': rank_followup_correct,
                    })

                for rank_followup_variable in range(rank_followup_context.H.shape[1]):
                    rank_followup_variable_rows.append({
                        'shot_id': rank_followup_case.shot_id,
                        'distance': rank_followup_context.distance,
                        'p': rank_followup_context.p,
                        'split': rank_followup_case.split,
                        'iteration': rank_followup_iteration,
                        'K_requested': k_requested,
                        'K': rank_followup_k,
                        'variable_id': rank_followup_variable,
                        'repair_relevant': int(rank_followup_repair[rank_followup_variable]),
                        'misranked': int(rank_followup_repair[rank_followup_variable] and rank_followup_ranks[rank_followup_variable] > rank_followup_k),
                        'final_rank': int(final_ranks[rank_followup_variable]),
                        **{name: float(values[rank_followup_variable]) for name, values in features.items()},
                    })

    repair_rank_trajectories = pd.DataFrame(repair_rows, columns=['shot_id', 'distance', 'p', 'split', 'iteration', 'relative_iteration', 'K_requested', 'K', 'Q_K', 'mean_repair_rank', 'max_repair_rank', 'osd_order', 'logical_correct'])
    misranked_variables = pd.DataFrame(rank_followup_variable_rows, columns=['shot_id', 'distance', 'p', 'split', 'iteration', 'K_requested', 'K', 'variable_id', 'repair_relevant', 'misranked', 'final_rank', 'current_abs_llr', 'message_instability', 'belief_instability', 'sign_flip', 'extrinsic_conflict', 'rank_volatility', 'sign_consistency', 'llr_variance', 'ranks'])
    repair_rank_trajectories.to_csv(repair_trajectory_path, index=False)
    misranked_variables.to_csv(misranked_path, index=False)

coverage_rows = []
for rank_followup_alignment, axis in [('absolute', 'iteration'), ('first_valid', 'relative_iteration')]:
    if len(repair_rank_trajectories):
        unique = repair_rank_trajectories.drop_duplicates(['shot_id', 'iteration', 'K'])
        for rank_followup_key, rank_followup_frame in unique.dropna(subset=[axis]).groupby(['distance', axis, 'K'], sort=True):
            q_mean, q_low, q_high = followup_bootstrap(rank_followup_frame.Q_K, len(coverage_rows))
            rank_mean, rank_low, rank_high = followup_bootstrap(rank_followup_frame.mean_repair_rank, 1000 + len(coverage_rows))
            coverage_rows.append({
                'alignment': rank_followup_alignment,
                'distance': rank_followup_key[0],
                'aligned_iteration': rank_followup_key[1],
                'K': rank_followup_key[2],
                'shots': rank_followup_frame.shot_id.nunique(),
                'mean_Q_K': q_mean,
                'Q_K_CI95_low': q_low,
                'Q_K_CI95_high': q_high,
                'mean_repair_rank': rank_mean,
                'repair_rank_CI95_low': rank_low,
                'repair_rank_CI95_high': rank_high,
                'mean_max_repair_rank': rank_followup_frame.max_repair_rank.mean(),
            })

repair_topk_coverage = pd.DataFrame(coverage_rows, columns=['alignment', 'distance', 'aligned_iteration', 'K', 'shots', 'mean_Q_K', 'Q_K_CI95_low', 'Q_K_CI95_high', 'mean_repair_rank', 'repair_rank_CI95_low', 'repair_rank_CI95_high', 'mean_max_repair_rank'])
repair_topk_coverage.to_csv(rank_followup_data_paths.output('repair_topk_coverage_csv'), index=False)
rank_success_rows = []

if len(repair_rank_trajectories):
    rank_success = repair_rank_trajectories.copy()
    rank_success['Q_bin'] = pd.cut(rank_success.Q_K, [-1e-12, 1e-12, 0.5, 1 - 1e-12, 1 + 1e-12], labels=['0', '(0,0.5]', '(0.5,1)', '1'], include_lowest=True)
    for rank_followup_key, rank_followup_frame in rank_success.groupby(['distance', 'K', 'osd_order', 'Q_bin'], observed=True, sort=True):
        rank_followup_mean, rank_followup_low, rank_followup_high = followup_bootstrap(rank_followup_frame.logical_correct, len(rank_success_rows))
        rank_success_rows.append({
            'distance': rank_followup_key[0],
            'K': rank_followup_key[1],
            'osd_order': rank_followup_key[2],
            'Q_bin': rank_followup_key[3],
            'evaluations': len(rank_followup_frame),
            'logical_correct': rank_followup_mean,
            'CI95_low': rank_followup_low,
            'CI95_high': rank_followup_high,
        })

repair_rank_vs_osd_success = pd.DataFrame(rank_success_rows, columns=['distance', 'K', 'osd_order', 'Q_bin', 'evaluations', 'logical_correct', 'CI95_low', 'CI95_high'])
repair_rank_vs_osd_success.to_csv(rank_followup_data_paths.output('repair_rank_vs_osd_success_csv'), index=False)
FEATURE_SETS = {
    'current_abs': ['current_abs_llr'],
    'plus_message': ['current_abs_llr', 'message_instability', 'belief_instability'],
    'plus_conflict': ['current_abs_llr', 'message_instability', 'belief_instability', 'sign_flip', 'extrinsic_conflict'],
    'plus_rank_volatility': ['current_abs_llr', 'rank_volatility'],
    'all_dynamics': ['current_abs_llr', 'message_instability', 'belief_instability', 'sign_flip', 'extrinsic_conflict', 'rank_volatility', 'sign_consistency', 'llr_variance'],
}

def fit_binary_model(train, test, features):
    train = train.dropna(subset=features + ['misranked'])
    test = test.dropna(subset=features + ['misranked'])
    if train.misranked.nunique() < 2 or not len(test):
        return (None, np.nan, np.nan)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced'))

    model.fit(train[features], train.misranked)
    probability = model.predict_proba(test[features])[:, 1]
    auc = roc_auc_score(test.misranked, probability) if test.misranked.nunique() == 2 else np.nan
    ap = average_precision_score(test.misranked, probability) if test.misranked.any() else np.nan

    return (model, float(auc), float(ap))

def grouped_cv_binary_metrics(frame, features):
    frame = frame.dropna(subset=features + ['misranked'])
    folds = min(FOLLOWUP_CONFIG['cv_folds'], frame.shot_id.nunique())
    if folds < 2 or frame.misranked.nunique() < 2:
        return (np.nan, np.nan, 0, 'insufficient_group_or_class_support')
    prediction = np.full(len(frame), np.nan)
    splitter = GroupKFold(n_splits=folds)

    for train_index, validation_index in splitter.split(frame[features], frame.misranked, frame.shot_id):
        train = frame.iloc[train_index]
        if train.misranked.nunique() < 2:
            continue
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced'))

        model.fit(train[features], train.misranked)
        prediction[validation_index] = model.predict_proba(frame.iloc[validation_index][features])[:, 1]

    evaluated = np.isfinite(prediction)
    labels = frame.misranked.to_numpy()[evaluated]
    if labels.size == 0 or np.unique(labels).size < 2:
        return (np.nan, np.nan, int(frame.loc[evaluated, 'shot_id'].nunique()), 'insufficient_fold_class_support')
    auc = roc_auc_score(labels, prediction[evaluated])
    ap = average_precision_score(labels, prediction[evaluated])

    return (float(auc), float(ap), int(frame.loc[evaluated, 'shot_id'].nunique()), 'complete')

ablation_rows = []
if len(misranked_variables):
    for rank_followup_key, all_variables in misranked_variables.groupby(['distance', 'K'], sort=True):
        repair_variables = all_variables[all_variables.repair_relevant == 1]
        train = repair_variables[repair_variables.split == 'development']
        test = repair_variables[repair_variables.split == 'test']

        for rank_followup_name, features in FEATURE_SETS.items():
            model, auc, ap = fit_binary_model(train, test, features)
            ablation_rows.append({
                'analysis_type': 'held_out_classifier',
                'distance': rank_followup_key[0],
                'K': rank_followup_key[1],
                'feature_set': rank_followup_name,
                'train_shots': train.shot_id.nunique(),
                'test_shots': test.shot_id.nunique(),
                'AUROC': auc,
                'AUPRC': ap,
                'spearman': np.nan,
                'comparison_group': np.nan,
                'mean_feature': np.nan,
                'median_feature': np.nan,
                'status': 'complete' if model is not None else 'insufficient_class_support',
            })
            cv_auc, cv_ap, cv_shots, cv_status = grouped_cv_binary_metrics(train, features)
            ablation_rows.append({
                'analysis_type': 'development_grouped_cv',
                'distance': rank_followup_key[0],
                'K': rank_followup_key[1],
                'feature_set': rank_followup_name,
                'train_shots': train.shot_id.nunique(),
                'test_shots': cv_shots,
                'AUROC': cv_auc,
                'AUPRC': cv_ap,
                'spearman': np.nan,
                'comparison_group': np.nan,
                'mean_feature': np.nan,
                'median_feature': np.nan,
                'status': cv_status,
            })

        for feature in FEATURE_SETS['all_dynamics']:
            scalar_model, scalar_auc, scalar_ap = fit_binary_model(train, test, [feature])
            feature_rank_correlation = rank_followup_safe_corr(all_variables[feature], all_variables.final_rank, 'spearman')
            ablation_rows.append({
                'analysis_type': 'held_out_scalar_predictor',
                'distance': rank_followup_key[0],
                'K': rank_followup_key[1],
                'feature_set': feature,
                'train_shots': train.shot_id.nunique(),
                'test_shots': test.shot_id.nunique(),
                'AUROC': scalar_auc,
                'AUPRC': scalar_ap,
                'spearman': feature_rank_correlation,
                'comparison_group': np.nan,
                'mean_feature': np.nan,
                'median_feature': np.nan,
                'status': 'complete' if scalar_model is not None else 'insufficient_class_support',
            })
            comparison_groups = {
                'non_repair': all_variables[all_variables.repair_relevant == 0],
                'repair_all': repair_variables,
                'repair_topK': repair_variables[repair_variables.misranked == 0],
                'repair_misranked': repair_variables[repair_variables.misranked == 1],
            }

            for rank_followup_label, subset in comparison_groups.items():
                ablation_rows.append({
                    'analysis_type': 'distribution',
                    'distance': rank_followup_key[0],
                    'K': rank_followup_key[1],
                    'feature_set': feature,
                    'train_shots': subset.shot_id.nunique(),
                    'test_shots': np.nan,
                    'AUROC': np.nan,
                    'AUPRC': np.nan,
                    'spearman': feature_rank_correlation,
                    'comparison_group': rank_followup_label,
                    'mean_feature': subset[feature].mean(),
                    'median_feature': subset[feature].median(),
                    'status': 'complete' if len(subset) else 'empty_group',
                })

misranking_feature_ablation = pd.DataFrame(ablation_rows, columns=['analysis_type', 'distance', 'K', 'feature_set', 'train_shots', 'test_shots', 'AUROC', 'AUPRC', 'spearman', 'comparison_group', 'mean_feature', 'median_feature', 'status'])
misranking_feature_ablation.to_csv(rank_followup_data_paths.output('misranking_feature_ablation_csv'), index=False)


## Experiments 3--4: oracle rank and snapshot ceilings

Construct oracle ceilings for ordering and snapshot choice separately. These quantify distinct opportunities and both use information that a runtime method cannot access.


In [ ]:
oracle_trial_path = rank_followup_data_paths.output('oracle_rank_trials_csv')
snapshot_oracle_path = rank_followup_data_paths.output('snapshot_oracle_trials_csv')
if followup_stage_cached([oracle_trial_path, snapshot_oracle_path]):
    oracle_rank_trials = rank_followup_read_csv(oracle_trial_path)
    snapshot_oracle_trials = rank_followup_read_csv(snapshot_oracle_path)
else:
    oracle_rows, snapshot_rows = ([], [])
    gap_cases = followup_cohort[followup_cohort.cohort_group == 'B'].drop_duplicates('shot_id')
    for rank_followup_case in tqdm(gap_cases.itertuples(index=False), total=len(gap_cases), desc='oracle rank and snapshot', disable=FOLLOWUP_CONFIG['print_level'] == 1):
        rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
        rank_followup_repair = rank_followup_bit_array(rank_followup_case.repair_support, rank_followup_context.H.shape[1]).astype(bool)
        final_order = np.argsort(np.abs(rank_followup_trace.llr[-1]), kind='stable')
        repair_ids = [int(v) for v in final_order if rank_followup_repair[v]]
        nonrepair_ids = [int(v) for v in final_order if not rank_followup_repair[v]]
        variants = {'baseline': final_order, 'repair_front': np.asarray(repair_ids + nonrepair_ids)}

        for k_requested in FOLLOWUP_CONFIG['top_k_values']:
            rank_followup_k = min(k_requested, len(final_order))
            if len(repair_ids) <= rank_followup_k:
                variants[f'minimal_top{k_requested}'] = minimal_topk_promotion(final_order, rank_followup_repair, rank_followup_k)

        for rank_followup_variable in repair_ids:
            variants[f'one_variable_{rank_followup_variable}'] = np.asarray([rank_followup_variable] + [v for v in final_order if v != rank_followup_variable])
        for variant, order_vector in variants.items():
            rank_followup_score = score_from_order(order_vector)
            for osd_order in FOLLOWUP_CONFIG['osd_orders_by_distance'][str(rank_followup_context.distance)]:
                rank_followup_correction = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_score, osd_order)
                oracle_rows.append({
                    'shot_id': rank_followup_case.shot_id,
                    'distance': rank_followup_context.distance,
                    'split': rank_followup_case.split,
                    'variant': variant,
                    'osd_order': osd_order,
                    'logical_correct': rank_followup_outcome(rank_followup_context, rank_followup_correction, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
                    'repair_support_size': int(rank_followup_repair.sum()),
                })

        previous_rank = None
        for rank_followup_iteration in range(1, len(rank_followup_trace.llr)):
            rank = rank_followup_ordinal_ranks(np.abs(rank_followup_trace.llr[rank_followup_iteration]))
            stability = np.nan if previous_rank is None else float(kendalltau(previous_rank, rank).statistic)
            previous_rank = rank
            residual = np.abs(rank_followup_trace.c2v[rank_followup_iteration] - rank_followup_trace.c2v[rank_followup_iteration - 1]).mean()

            for osd_order in [0, 1]:
                rank_followup_correction = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_trace.llr[rank_followup_iteration], osd_order)
                snapshot_rows.append({
                    'shot_id': rank_followup_case.shot_id,
                    'distance': rank_followup_context.distance,
                    'split': rank_followup_case.split,
                    'iteration': rank_followup_iteration,
                    'osd_order': osd_order,
                    'logical_correct': rank_followup_outcome(rank_followup_context, rank_followup_correction, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
                    'message_residual': residual,
                    'rank_stability': stability,
                    'syndrome_consistent': int(rank_followup_trace.valid[rank_followup_iteration]),
                })

    oracle_rank_trials = pd.DataFrame(oracle_rows, columns=['shot_id', 'distance', 'split', 'variant', 'osd_order', 'logical_correct', 'repair_support_size'])
    snapshot_oracle_trials = pd.DataFrame(snapshot_rows, columns=['shot_id', 'distance', 'split', 'iteration', 'osd_order', 'logical_correct', 'message_residual', 'rank_stability', 'syndrome_consistent'])
    oracle_rank_trials.to_csv(oracle_trial_path, index=False)
    snapshot_oracle_trials.to_csv(snapshot_oracle_path, index=False)

oracle_rows = []
if len(oracle_rank_trials):
    rank_followup_baseline = oracle_rank_trials[oracle_rank_trials.variant == 'baseline'][['shot_id', 'osd_order', 'logical_correct']].rename(columns={'logical_correct': 'baseline_correct'})
    rank_followup_paired = oracle_rank_trials.merge(rank_followup_baseline, on=['shot_id', 'osd_order'], validate='many_to_one')
    rank_followup_paired['paired_gain'] = rank_followup_paired.logical_correct - rank_followup_paired.baseline_correct

    for rank_followup_key, rank_followup_frame in rank_followup_paired.groupby(['distance', 'variant', 'osd_order'], sort=True):
        success, rank_followup_low, rank_followup_high = followup_bootstrap(rank_followup_frame.logical_correct, len(oracle_rows))
        gain, gain_low, gain_high = followup_bootstrap(rank_followup_frame.paired_gain, 1000 + len(oracle_rows))
        oracle_rows.append({
            'distance': rank_followup_key[0],
            'variant': rank_followup_key[1],
            'osd_order': rank_followup_key[2],
            'shots': rank_followup_frame.shot_id.nunique(),
            'logical_correct': success,
            'CI95_low': rank_followup_low,
            'CI95_high': rank_followup_high,
            'paired_gain': gain,
            'gain_CI95_low': gain_low,
            'gain_CI95_high': gain_high,
        })

oracle_rank_ceiling = pd.DataFrame(oracle_rows, columns=['distance', 'variant', 'osd_order', 'shots', 'logical_correct', 'CI95_low', 'CI95_high', 'paired_gain', 'gain_CI95_low', 'gain_CI95_high'])
oracle_rank_ceiling.to_csv(rank_followup_data_paths.output('oracle_rank_ceiling_csv'), index=False)
snapshot_rows = []

if len(snapshot_oracle_trials):
    for (rank_followup_distance, rank_followup_order), rank_followup_frame in snapshot_oracle_trials.groupby(['distance', 'osd_order'], sort=True):
        rank_followup_selected = []
        for rank_followup_shot_id, shot in rank_followup_frame.groupby('shot_id', sort=True):
            rules = {
                'final': shot.iloc[-1],
                'first_valid': shot[shot.syndrome_consistent == 1].iloc[0] if (shot.syndrome_consistent == 1).any() else shot.iloc[-1],
                'min_message_residual': shot.loc[shot.message_residual.idxmin()],
                'max_rank_stability': shot.dropna(subset=['rank_stability']).sort_values(['rank_stability', 'iteration'], ascending=[False, True]).iloc[0] if shot.rank_stability.notna().any() else shot.iloc[-1],
                'best_snapshot_oracle': shot.sort_values(['logical_correct', 'iteration'], ascending=[False, True]).iloc[0],
            }

            for rank_followup_rule, rank_followup_row in rules.items():
                rank_followup_selected.append({
                    'shot_id': rank_followup_shot_id,
                    'rule': rank_followup_rule,
                    'logical_correct': rank_followup_row.logical_correct,
                    'iteration': rank_followup_row.iteration,
                })

        rank_followup_selected = pd.DataFrame(rank_followup_selected)
        final = rank_followup_selected[rank_followup_selected.rule == 'final'][['shot_id', 'logical_correct']].rename(columns={'logical_correct': 'final_correct'})
        rank_followup_selected = rank_followup_selected.merge(final, on='shot_id', validate='many_to_one')
        rank_followup_selected['paired_gain'] = rank_followup_selected.logical_correct - rank_followup_selected.final_correct

        for rank_followup_rule, rule_frame in rank_followup_selected.groupby('rule', sort=True):
            success, rank_followup_low, rank_followup_high = followup_bootstrap(rule_frame.logical_correct, len(snapshot_rows))
            gain, gain_low, gain_high = followup_bootstrap(rule_frame.paired_gain, 1000 + len(snapshot_rows))
            snapshot_rows.append({
                'distance': rank_followup_distance,
                'osd_order': rank_followup_order,
                'rule': rank_followup_rule,
                'shots': rule_frame.shot_id.nunique(),
                'logical_correct': success,
                'CI95_low': rank_followup_low,
                'CI95_high': rank_followup_high,
                'paired_gain_vs_final': gain,
                'gain_CI95_low': gain_low,
                'gain_CI95_high': gain_high,
                'mean_selected_iteration': rule_frame.iteration.mean(),
            })

snapshot_oracle_ceiling = pd.DataFrame(snapshot_rows, columns=['distance', 'osd_order', 'rule', 'shots', 'logical_correct', 'CI95_low', 'CI95_high', 'paired_gain_vs_final', 'gain_CI95_low', 'gain_CI95_high', 'mean_selected_iteration'])
snapshot_oracle_ceiling.to_csv(rank_followup_data_paths.output('snapshot_oracle_ceiling_csv'), index=False)


## Experiment 5: runtime snapshot selection

Select a snapshot using runtime features and evaluate it against endpoint and oracle references. A good snapshot existing in the trace does not imply that the practical rule can recognize it.


In [ ]:
snapshot_feature_path = rank_followup_data_paths.output('snapshot_runtime_features_csv')
if followup_stage_cached([snapshot_feature_path]):
    snapshot_runtime_features = rank_followup_read_csv(snapshot_feature_path)
else:
    rank_followup_rows = []
    gap_cases = followup_cohort[followup_cohort.cohort_group == 'B'].drop_duplicates('shot_id')
    for rank_followup_case in tqdm(gap_cases.itertuples(index=False), total=len(gap_cases), desc='runtime snapshot features', disable=FOLLOWUP_CONFIG['print_level'] == 1):
        rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
        previous_rank = rank_followup_ordinal_ranks(np.abs(rank_followup_trace.llr[0]))
        rank_followup_k = min(10, rank_followup_context.H.shape[1])

        for rank_followup_iteration in range(1, len(rank_followup_trace.llr)):
            features = prefix_features(rank_followup_context, rank_followup_trace, rank_followup_iteration)
            rank = features['ranks']
            rank_stability = float(kendalltau(previous_rank, rank).statistic)
            topk_stability = len(set(np.flatnonzero(previous_rank <= rank_followup_k)) & set(np.flatnonzero(rank <= rank_followup_k))) / len(set(np.flatnonzero(previous_rank <= rank_followup_k)) | set(np.flatnonzero(rank <= rank_followup_k)))
            previous_rank = rank
            hard = rank_followup_trace.hard[rank_followup_iteration]
            unsatisfied_checks = int(np.sum(rank_followup_context.H @ hard % 2 != rank_followup_syndrome))
            rank_followup_correction = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_trace.llr[rank_followup_iteration], 1)
            rank_followup_rows.append({
                'shot_id': rank_followup_case.shot_id,
                'distance': rank_followup_context.distance,
                'split': rank_followup_case.split,
                'iteration': rank_followup_iteration,
                'logical_correct': rank_followup_outcome(rank_followup_context, rank_followup_correction, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
                'syndrome_consistent': int(rank_followup_trace.valid[rank_followup_iteration]),
                'message_residual': float(np.abs(rank_followup_trace.c2v[rank_followup_iteration] - rank_followup_trace.c2v[rank_followup_iteration - 1]).mean()),
                'rank_stability': rank_stability,
                'topk_stability': topk_stability,
                'mean_rank_volatility': float(features['rank_volatility'].mean()),
                'mean_conflict': float(features['extrinsic_conflict'].mean()),
                'minimum_confidence': float(features['current_abs_llr'].min()),
                'q10_confidence': float(np.quantile(features['current_abs_llr'], 0.1)),
                'unsatisfied_checks': unsatisfied_checks,
            })

    snapshot_runtime_features = pd.DataFrame(rank_followup_rows, columns=['shot_id', 'distance', 'split', 'iteration', 'logical_correct', 'syndrome_consistent', 'message_residual', 'rank_stability', 'topk_stability', 'mean_rank_volatility', 'mean_conflict', 'minimum_confidence', 'q10_confidence', 'unsatisfied_checks'])
    snapshot_runtime_features.to_csv(snapshot_feature_path, index=False)

SNAPSHOT_FEATURES = ['message_residual', 'rank_stability', 'topk_stability', 'mean_rank_volatility', 'mean_conflict', 'minimum_confidence', 'q10_confidence', 'unsatisfied_checks', 'iteration']
snapshot_rule_rows = []
if len(snapshot_runtime_features):
    for rank_followup_distance, rank_followup_frame in snapshot_runtime_features.groupby('distance', sort=True):
        rank_followup_development = rank_followup_frame[rank_followup_frame.split == 'development'].dropna(subset=SNAPSHOT_FEATURES)
        test = rank_followup_frame[rank_followup_frame.split == 'test'].dropna(subset=SNAPSHOT_FEATURES)
        models = {}

        if rank_followup_development.logical_correct.nunique() == 2:
            models['logistic'] = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced'))
            models['tree'] = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=FOLLOWUP_CONFIG['random_seed'])
            for model in models.values():
                model.fit(rank_followup_development[SNAPSHOT_FEATURES], rank_followup_development.logical_correct)

        residual_threshold = rank_followup_development.message_residual.quantile(0.25) if len(rank_followup_development) else np.nan
        stability_threshold = rank_followup_development.rank_stability.quantile(0.75) if len(rank_followup_development) else np.nan
        rank_followup_selected_rows = []

        for rank_followup_shot_id, shot in test.groupby('shot_id', sort=True):
            rules = {
                'final': shot.iloc[-1],
                'first_valid': shot[shot.syndrome_consistent == 1].iloc[0] if (shot.syndrome_consistent == 1).any() else shot.iloc[-1],
                'min_message_residual': shot.loc[shot.message_residual.idxmin()],
                'max_rank_stability': shot.loc[shot.rank_stability.idxmax()],
            }

            practical = shot[(shot.message_residual <= residual_threshold) & (shot.rank_stability >= stability_threshold)]
            rules['practical_threshold'] = practical.iloc[0] if len(practical) else shot.iloc[-1]
            for model_name, model in models.items():
                rank_followup_probability = model.predict_proba(shot[SNAPSHOT_FEATURES])[:, 1]
                rules[model_name] = shot.iloc[int(np.argmax(rank_followup_probability))]

            for rank_followup_rule, rank_followup_selected in rules.items():
                rank_followup_selected_rows.append({
                    'shot_id': rank_followup_shot_id,
                    'rule': rank_followup_rule,
                    'logical_correct': rank_followup_selected.logical_correct,
                    'iteration': rank_followup_selected.iteration,
                })

        if rank_followup_selected_rows:
            rank_followup_selected = pd.DataFrame(rank_followup_selected_rows)
            final = rank_followup_selected[rank_followup_selected.rule == 'final'][['shot_id', 'logical_correct']].rename(columns={'logical_correct': 'final_correct'})
            rank_followup_selected = rank_followup_selected.merge(final, on='shot_id', validate='many_to_one')
            rank_followup_selected['repair_vs_final'] = ((rank_followup_selected.final_correct == 0) & (rank_followup_selected.logical_correct == 1)).astype(int)
            rank_followup_selected['regression_vs_final'] = ((rank_followup_selected.final_correct == 1) & (rank_followup_selected.logical_correct == 0)).astype(int)

            for rank_followup_rule, rule_frame in rank_followup_selected.groupby('rule', sort=True):
                rank_followup_mean, rank_followup_low, rank_followup_high = followup_bootstrap(rule_frame.logical_correct, len(snapshot_rule_rows))
                snapshot_rule_rows.append({
                    'distance': rank_followup_distance,
                    'rule': rank_followup_rule,
                    'test_shots': rule_frame.shot_id.nunique(),
                    'logical_correct': rank_followup_mean,
                    'CI95_low': rank_followup_low,
                    'CI95_high': rank_followup_high,
                    'repair_rate': rule_frame.repair_vs_final.mean(),
                    'regression_rate': rule_frame.regression_vs_final.mean(),
                    'mean_selected_iteration': rule_frame.iteration.mean(),
                    'model_status': 'complete' if rank_followup_rule in models or rank_followup_rule not in {'logistic', 'tree'} else 'insufficient_class_support',
                })

snapshot_rule_benchmark = pd.DataFrame(snapshot_rule_rows, columns=['distance', 'rule', 'test_shots', 'logical_correct', 'CI95_low', 'CI95_high', 'repair_rate', 'regression_rate', 'mean_selected_iteration', 'model_status'])
snapshot_rule_benchmark.to_csv(rank_followup_data_paths.output('snapshot_rule_benchmark_csv'), index=False)


## Experiment 6: held-out dynamics-aware rank correction

Fit and evaluate the dynamics-aware rank correction on held-out inputs. Keep snapshots from a shared input grouped so related observations do not supply an artificial generalization gain.


In [ ]:
rank_correction_path = rank_followup_data_paths.output('rank_correction_trials_csv')
rank_parameter_path = rank_followup_data_paths.output('rank_correction_selected_parameters_csv')

def standardize_vector(values):
    values = np.asarray(values, dtype=float)
    scale = values.std()
    return (values - values.mean()) / (scale if scale > 0 else 1.0)

def corrected_rank_score(features, weights):
    penalties = [features['message_instability'], features['rank_volatility'], features['extrinsic_conflict'], 1 - features['sign_consistency']]
    return standardize_vector(features['current_abs_llr']) - sum((float(weight) * standardize_vector(value) for weight, value in zip(weights, penalties)))

if followup_stage_cached([rank_correction_path, rank_parameter_path]):
    rank_correction_trials = rank_followup_read_csv(rank_correction_path)
    rank_correction_parameters = pd.read_csv(rank_parameter_path)
else:
    gap_cases = followup_cohort[followup_cohort.cohort_group == 'B'].drop_duplicates('shot_id')
    parameter_rows = []
    for rank_followup_distance, rank_followup_frame in gap_cases[gap_cases.split == 'development'].groupby('distance', sort=True):
        rank_followup_candidates = list(product(FOLLOWUP_CONFIG['rank_weight_values'], repeat=4))
        objective_rows = []
        for rank_followup_weights in tqdm(rank_followup_candidates, desc=f'rank correction tuning d={rank_followup_distance}', disable=FOLLOWUP_CONFIG['print_level'] == 1):
            rank_followup_scores = []
            for rank_followup_case in rank_followup_frame.itertuples(index=False):
                rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
                rank_followup_repair = rank_followup_bit_array(rank_followup_case.repair_support, rank_followup_context.H.shape[1]).astype(bool)
                features = prefix_features(rank_followup_context, rank_followup_trace, len(rank_followup_trace.llr) - 1)
                rank_followup_ranks = rank_followup_ordinal_ranks(corrected_rank_score(features, rank_followup_weights))
                rank_followup_k = min(10, len(rank_followup_ranks))
                rank_followup_scores.append(np.mean(rank_followup_ranks[rank_followup_repair] <= rank_followup_k))

            objective_rows.append((float(np.mean(rank_followup_scores)), rank_followup_weights))

        best_objective, best_weights = sorted(objective_rows, key=lambda item: (-item[0], item[1]))[0]
        parameter_rows.append({
            'distance': rank_followup_distance,
            'objective': 'development_mean_Q10',
            'objective_value': best_objective,
            **{f'beta_{name}': value for name, value in zip(['message', 'rank', 'conflict', 'sign'], best_weights)},
        })

    rank_correction_parameters = pd.DataFrame(parameter_rows, columns=['distance', 'objective', 'objective_value', 'beta_message', 'beta_rank', 'beta_conflict', 'beta_sign'])
    rank_correction_parameters.to_csv(rank_parameter_path, index=False)
    rank_followup_rows = []
    parameter_lookup = {int(row.distance): tuple((float(getattr(row, f'beta_{name}')) for name in ['message', 'rank', 'conflict', 'sign'])) for row in rank_correction_parameters.itertuples(index=False)}

    for rank_followup_case in tqdm(gap_cases[gap_cases.split == 'test'].itertuples(index=False), total=(gap_cases.split == 'test').sum(), desc='rank correction held-out', disable=FOLLOWUP_CONFIG['print_level'] == 1):
        rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
        rank_followup_repair = rank_followup_bit_array(rank_followup_case.repair_support, rank_followup_context.H.shape[1]).astype(bool)
        features = prefix_features(rank_followup_context, rank_followup_trace, len(rank_followup_trace.llr) - 1)
        rank_followup_baseline = features['current_abs_llr']
        rank_followup_weights = parameter_lookup.get(rank_followup_context.distance, (0.0, 0.0, 0.0, 0.0))
        dynamic = corrected_rank_score(features, rank_followup_weights)
        baseline_order = np.argsort(rank_followup_baseline, kind='stable')
        repair_ids = [int(v) for v in baseline_order if rank_followup_repair[v]]
        oracle = score_from_order(np.asarray(repair_ids + [int(v) for v in baseline_order if not rank_followup_repair[v]]))

        for rank_followup_method, rank_followup_score in {
            'baseline_final': rank_followup_baseline,
            'dynamic_rank': dynamic,
            'oracle_repair_front': oracle,
        }.items():
            rank_followup_ranks = rank_followup_ordinal_ranks(rank_followup_score)
            for osd_order in [0, 1]:
                rank_followup_correction = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_score, osd_order)
                rank_followup_rows.append({
                    'shot_id': rank_followup_case.shot_id,
                    'distance': rank_followup_context.distance,
                    'split': rank_followup_case.split,
                    'method': rank_followup_method,
                    'osd_order': osd_order,
                    'logical_correct': rank_followup_outcome(rank_followup_context, rank_followup_correction, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
                    'Q10': float(np.mean(rank_followup_ranks[rank_followup_repair] <= min(10, len(rank_followup_ranks)))),
                    'mean_repair_rank': float(rank_followup_ranks[rank_followup_repair].mean()),
                })

        shot_snapshots = snapshot_oracle_trials[(snapshot_oracle_trials.shot_id == rank_followup_case.shot_id) & snapshot_oracle_trials.osd_order.isin([0, 1])]
        for rank_followup_order, order_frame in shot_snapshots.groupby('osd_order'):
            best = int(order_frame.logical_correct.max())
            rank_followup_rows.append({
                'shot_id': rank_followup_case.shot_id,
                'distance': rank_followup_context.distance,
                'split': rank_followup_case.split,
                'method': 'best_snapshot_oracle',
                'osd_order': int(rank_followup_order),
                'logical_correct': best,
                'Q10': np.nan,
                'mean_repair_rank': np.nan,
            })

    rank_correction_trials = pd.DataFrame(rank_followup_rows, columns=['shot_id', 'distance', 'split', 'method', 'osd_order', 'logical_correct', 'Q10', 'mean_repair_rank'])
    rank_correction_trials.to_csv(rank_correction_path, index=False)

rank_followup_rank_rows = []
if len(rank_correction_trials):
    rank_followup_baseline = rank_correction_trials[rank_correction_trials.method == 'baseline_final'][['shot_id', 'osd_order', 'logical_correct']].rename(columns={'logical_correct': 'baseline_correct'})
    rank_followup_paired = rank_correction_trials.merge(rank_followup_baseline, on=['shot_id', 'osd_order'], validate='many_to_one')
    rank_followup_paired['gain'] = rank_followup_paired.logical_correct - rank_followup_paired.baseline_correct

    for rank_followup_key, rank_followup_frame in rank_followup_paired.groupby(['distance', 'method', 'osd_order'], sort=True):
        success, rank_followup_low, rank_followup_high = followup_bootstrap(rank_followup_frame.logical_correct, len(rank_followup_rank_rows))
        gain, gain_low, gain_high = followup_bootstrap(rank_followup_frame.gain, 1000 + len(rank_followup_rank_rows))
        rank_followup_rank_rows.append({
            'distance': rank_followup_key[0],
            'method': rank_followup_key[1],
            'osd_order': rank_followup_key[2],
            'test_shots': rank_followup_frame.shot_id.nunique(),
            'logical_correct': success,
            'CI95_low': rank_followup_low,
            'CI95_high': rank_followup_high,
            'paired_gain': gain,
            'gain_CI95_low': gain_low,
            'gain_CI95_high': gain_high,
            'mean_Q10': rank_followup_frame.Q10.mean(),
            'mean_repair_rank': rank_followup_frame.mean_repair_rank.mean(),
        })

rank_correction_ablation = pd.DataFrame(rank_followup_rank_rows, columns=['distance', 'method', 'osd_order', 'test_shots', 'logical_correct', 'CI95_low', 'CI95_high', 'paired_gain', 'gain_CI95_low', 'gain_CI95_high', 'mean_Q10', 'mean_repair_rank'])
rank_correction_ablation.to_csv(rank_followup_data_paths.output('rank_correction_ablation_csv'), index=False)


## Experiment 7: pairwise repair-variable ranking

Use pairwise ranking to test whether repair variables can be ordered more usefully than with scalar reliability alone. Compare actual OSD outcomes alongside rank-quality summaries.


In [ ]:
pairwise_trial_path = rank_followup_data_paths.output('pairwise_rank_trials_csv')
PAIR_FEATURES = ['current_abs_llr', 'message_instability', 'belief_instability', 'sign_flip', 'extrinsic_conflict', 'rank_volatility', 'sign_consistency', 'llr_variance']
if followup_stage_cached([pairwise_trial_path]):
    pairwise_rank_trials = rank_followup_read_csv(pairwise_trial_path)
else:
    gap_cases = followup_cohort[followup_cohort.cohort_group == 'B'].drop_duplicates('shot_id')
    rank_followup_variable_rows, pair_rows = ([], [])
    rank_followup_rng = np.random.default_rng(FOLLOWUP_CONFIG['random_seed'])

    for rank_followup_case in tqdm(gap_cases.itertuples(index=False), total=len(gap_cases), desc='pairwise rank data', disable=FOLLOWUP_CONFIG['print_level'] == 1):
        rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
        rank_followup_repair = rank_followup_bit_array(rank_followup_case.repair_support, rank_followup_context.H.shape[1]).astype(bool)
        features = prefix_features(rank_followup_context, rank_followup_trace, len(rank_followup_trace.llr) - 1)

        for rank_followup_variable in range(rank_followup_context.H.shape[1]):
            rank_followup_variable_rows.append({
                'shot_id': rank_followup_case.shot_id,
                'distance': rank_followup_context.distance,
                'split': rank_followup_case.split,
                'variable_id': rank_followup_variable,
                'repair_relevant': int(rank_followup_repair[rank_followup_variable]),
                **{name: features[name][rank_followup_variable] for name in PAIR_FEATURES},
            })

        repair_ids, other_ids = (np.flatnonzero(rank_followup_repair), np.flatnonzero(~rank_followup_repair))
        pairs = np.asarray(list(product(repair_ids, other_ids)), dtype=int)
        if len(pairs) > FOLLOWUP_CONFIG['pair_limit_per_shot']:
            pairs = pairs[rank_followup_rng.choice(len(pairs), FOLLOWUP_CONFIG['pair_limit_per_shot'], replace=False)]

        for repair_id, other_id in pairs:
            difference = {name: features[name][repair_id] - features[name][other_id] for name in PAIR_FEATURES}
            pair_rows.append({
                'shot_id': rank_followup_case.shot_id,
                'distance': rank_followup_context.distance,
                'split': rank_followup_case.split,
                'target_left_repair': 1,
                **difference,
            })
            pair_rows.append({
                'shot_id': rank_followup_case.shot_id,
                'distance': rank_followup_context.distance,
                'split': rank_followup_case.split,
                'target_left_repair': 0,
                **{name: -value for name, value in difference.items()},
            })

    variable_frame, pair_frame = (pd.DataFrame(rank_followup_variable_rows), pd.DataFrame(pair_rows))
    rank_followup_rows = []
    for rank_followup_distance, rank_followup_frame in pair_frame.groupby('distance', sort=True) if len(pair_frame) else []:
        train, test = (rank_followup_frame[rank_followup_frame.split == 'development'], rank_followup_frame[rank_followup_frame.split == 'test'])
        if len(train) and len(test) and (train.target_left_repair.nunique() == 2):
            model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
            model.fit(train[PAIR_FEATURES], train.target_left_repair)
            accuracy = float((model.predict(test[PAIR_FEATURES]) == test.target_left_repair).mean())
            rank_followup_rows.append({
                'shot_id': '__summary__',
                'distance': rank_followup_distance,
                'split': 'test',
                'method': 'pairwise_accuracy',
                'osd_order': -1,
                'logical_correct': np.nan,
                'top10_coverage': np.nan,
                'pairwise_accuracy': accuracy,
            })
            variables = variable_frame[(variable_frame.distance == rank_followup_distance) & (variable_frame.split == 'test')]

            for rank_followup_shot_id, shot in variables.groupby('shot_id', sort=True):
                rank_followup_case = gap_cases[gap_cases.shot_id == rank_followup_shot_id].iloc[0]
                rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
                rank_followup_repair = shot.repair_relevant.to_numpy(dtype=bool)
                rank_followup_score = -model.decision_function(shot[PAIR_FEATURES])
                rank_followup_ranks = rank_followup_ordinal_ranks(rank_followup_score)

                for rank_followup_order in [0, 1]:
                    rank_followup_correction = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_score, rank_followup_order)
                    rank_followup_rows.append({
                        'shot_id': rank_followup_shot_id,
                        'distance': rank_followup_distance,
                        'split': 'test',
                        'method': 'pairwise_model',
                        'osd_order': rank_followup_order,
                        'logical_correct': rank_followup_outcome(rank_followup_context, rank_followup_correction, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
                        'top10_coverage': np.mean(rank_followup_ranks[rank_followup_repair] <= min(10, len(rank_followup_ranks))),
                        'pairwise_accuracy': accuracy,
                    })

    pairwise_rank_trials = pd.DataFrame(rank_followup_rows, columns=['shot_id', 'distance', 'split', 'method', 'osd_order', 'logical_correct', 'top10_coverage', 'pairwise_accuracy'])
    pairwise_rank_trials.to_csv(pairwise_trial_path, index=False)

pairwise_rows = []
if len(pairwise_rank_trials):
    for rank_followup_key, rank_followup_frame in pairwise_rank_trials.groupby(['distance', 'method', 'osd_order'], sort=True):
        pairwise_rows.append({
            'distance': rank_followup_key[0],
            'method': rank_followup_key[1],
            'osd_order': rank_followup_key[2],
            'test_shots': rank_followup_frame[rank_followup_frame.shot_id != '__summary__'].shot_id.nunique(),
            'pairwise_accuracy': rank_followup_frame.pairwise_accuracy.mean(),
            'mean_top10_coverage': rank_followup_frame.top10_coverage.mean(),
            'logical_correct': rank_followup_frame.logical_correct.mean(),
        })

pairwise_rank_model = pd.DataFrame(pairwise_rows, columns=['distance', 'method', 'osd_order', 'test_shots', 'pairwise_accuracy', 'mean_top10_coverage', 'logical_correct'])
pairwise_rank_model.to_csv(rank_followup_data_paths.output('pairwise_rank_model_csv'), index=False)


## Experiment 8: BP parameter sensitivity of rank quality

Change the BP soft-information generator and measure resulting rank quality. Preserve the configured OSD search conditions when attributing a gain to BP parameters.


In [ ]:
parameter_trial_path = rank_followup_data_paths.output('bp_parameter_rank_trials_csv')
if followup_stage_cached([parameter_trial_path]):
    bp_parameter_rank_trials = rank_followup_read_csv(parameter_trial_path)
else:
    rank_followup_rows = []
    gap_cases = followup_cohort[followup_cohort.cohort_group == 'B'].drop_duplicates('shot_id')
    rank_followup_grid = list(product(FOLLOWUP_CONFIG['scaling_values'], FOLLOWUP_CONFIG['damping_values']))

    for rank_followup_case in tqdm(gap_cases.itertuples(index=False), total=len(gap_cases), desc='BP parameter rank quality', disable=FOLLOWUP_CONFIG['print_level'] == 1):
        rank_followup_context = rank_followup_context_for(int(rank_followup_case.level), float(rank_followup_case.p))
        rank_followup_syndrome = rank_followup_bit_array(rank_followup_case.syndrome, rank_followup_context.H.shape[0])
        rank_followup_truth = rank_followup_bit_array(rank_followup_case.true_logical, rank_followup_context.L.shape[0])
        rank_followup_error = rank_followup_bit_array(rank_followup_case.error, rank_followup_context.H.shape[1])
        baseline_trace = followup_case_material(rank_followup_case)[-1]
        baseline_rank = rank_followup_ordinal_ranks(np.abs(baseline_trace.llr[-1]))
        rank_followup_repair = rank_followup_bit_array(rank_followup_case.repair_support, rank_followup_context.H.shape[1]).astype(bool)

        for rank_followup_scaling, rank_followup_damping in rank_followup_grid:
            rank_followup_trace = followup_case_material(rank_followup_case, rank_followup_scaling, rank_followup_damping)[-1]
            rank_followup_endpoint = first_valid_or_final(rank_followup_trace)
            rank = rank_followup_ordinal_ranks(np.abs(rank_followup_trace.llr[rank_followup_endpoint]))
            rank_followup_k = min(10, len(rank))
            rank_followup_bp_correct = rank_followup_outcome(rank_followup_context, rank_followup_trace.hard[rank_followup_endpoint], rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1]
            left = set(np.flatnonzero(baseline_rank <= rank_followup_k))
            right = set(np.flatnonzero(rank <= rank_followup_k))
            rank_followup_common = {
                'shot_id': rank_followup_case.shot_id,
                'distance': rank_followup_context.distance,
                'p': rank_followup_context.p,
                'split': rank_followup_case.split,
                'scaling': rank_followup_scaling,
                'damping': rank_followup_damping,
                'schedule': 'parallel',
                'endpoint_iteration': rank_followup_endpoint,
                'bp_logical_correct': rank_followup_bp_correct,
                'Q10': float(np.mean(rank[rank_followup_repair] <= rank_followup_k)),
                'mean_repair_rank': float(rank[rank_followup_repair].mean()),
                'kendall_vs_default': float(kendalltau(baseline_rank, rank).statistic),
                'top10_jaccard_vs_default': len(left & right) / len(left | right),
            }

            for rank_followup_order in [0, 1]:
                rank_followup_correction = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_trace.llr[rank_followup_endpoint], rank_followup_order)
                rank_followup_rows.append(rank_followup_common | {
                    'osd_order': rank_followup_order,
                    'osd_logical_correct': rank_followup_outcome(rank_followup_context, rank_followup_correction, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
                })

    bp_parameter_rank_trials = pd.DataFrame(rank_followup_rows, columns=['shot_id', 'distance', 'p', 'split', 'scaling', 'damping', 'schedule', 'endpoint_iteration', 'bp_logical_correct', 'Q10', 'mean_repair_rank', 'kendall_vs_default', 'top10_jaccard_vs_default', 'osd_order', 'osd_logical_correct'])
    bp_parameter_rank_trials.to_csv(parameter_trial_path, index=False)

parameter_rows = []
if len(bp_parameter_rank_trials):
    for rank_followup_key, rank_followup_frame in bp_parameter_rank_trials.groupby(['distance', 'scaling', 'damping', 'schedule', 'osd_order'], sort=True):
        success, rank_followup_low, rank_followup_high = followup_bootstrap(rank_followup_frame.osd_logical_correct, len(parameter_rows))
        parameter_rows.append({
            'distance': rank_followup_key[0],
            'scaling': rank_followup_key[1],
            'damping': rank_followup_key[2],
            'schedule': rank_followup_key[3],
            'osd_order': rank_followup_key[4],
            'shots': rank_followup_frame.shot_id.nunique(),
            'BP_LER': 1 - rank_followup_frame.bp_logical_correct.mean(),
            'OSD_LER': 1 - success,
            'OSD_LER_CI95_low': 1 - rank_followup_high,
            'OSD_LER_CI95_high': 1 - rank_followup_low,
            'mean_Q10': rank_followup_frame.Q10.mean(),
            'mean_repair_rank': rank_followup_frame.mean_repair_rank.mean(),
            'mean_kendall_vs_default': rank_followup_frame.kendall_vs_default.mean(),
            'mean_top10_jaccard_vs_default': rank_followup_frame.top10_jaccard_vs_default.mean(),
        })

bp_parameter_rank_quality = pd.DataFrame(parameter_rows, columns=['distance', 'scaling', 'damping', 'schedule', 'osd_order', 'shots', 'BP_LER', 'OSD_LER', 'OSD_LER_CI95_low', 'OSD_LER_CI95_high', 'mean_Q10', 'mean_repair_rank', 'mean_kendall_vs_default', 'mean_top10_jaccard_vs_default'])
bp_parameter_rank_quality.to_csv(rank_followup_data_paths.output('bp_parameter_rank_quality_csv'), index=False)


## Experiment 9: trace-native final-gap benchmark

Gap closure is emitted only when the paired bootstrap 95% lower bound of the
standard-trace minus Tesseract LER gap is positive.

Compare frozen methods on the separate gap-regime population. Report practical results alongside oracle ceilings and keep regressions on baseline successes in the LER.


In [ ]:
final_gap_trial_path = rank_followup_data_paths.output('final_gap_trials_csv')
if followup_stage_cached([final_gap_trial_path]):
    final_gap_trials = rank_followup_read_csv(final_gap_trial_path)
else:
    rank_followup_rows = []
    parameter_lookup = {int(row.distance): tuple((float(getattr(row, f'beta_{name}')) for name in ['message', 'rank', 'conflict', 'sign'])) for row in rank_correction_parameters.itertuples(index=False)}
    for rank_followup_case in tqdm(rank_followup_unbiased_cases.itertuples(index=False), total=len(rank_followup_unbiased_cases), desc='trace-native final gap', disable=FOLLOWUP_CONFIG['print_level'] == 1):
        rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
        bundle = traced_decode_bundle(rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace)
        features = prefix_features(rank_followup_context, rank_followup_trace, len(rank_followup_trace.llr) - 1)
        full_order = FOLLOWUP_CONFIG['osd_orders_by_distance'][str(rank_followup_context.distance)][-1]
        forced_osd1 = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_trace.llr[-1], 1)
        full_correction = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_trace.llr[-1], full_order)
        dynamic_score = corrected_rank_score(features, parameter_lookup.get(rank_followup_context.distance, (0.0, 0.0, 0.0, 0.0)))
        dynamic_osd1 = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, dynamic_score, 1)
        rank_followup_repair = bundle['bp'] ^ full_correction
        baseline_order = np.argsort(np.abs(rank_followup_trace.llr[-1]), kind='stable')
        oracle_order = np.asarray([int(v) for v in baseline_order if rank_followup_repair[v]] + [int(v) for v in baseline_order if not rank_followup_repair[v]])
        oracle_osd1 = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, score_from_order(oracle_order), 1)
        snapshot_results = []

        for rank_followup_iteration in range(1, len(rank_followup_trace.llr)):
            rank_followup_correction = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, rank_followup_trace.llr[rank_followup_iteration], 1)
            snapshot_results.append((rank_followup_outcome(rank_followup_context, rank_followup_correction, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1], rank_followup_iteration))

        best_correct, best_iteration = sorted(snapshot_results, key=lambda item: (-item[0], item[1]))[0]
        methods = {
            'standard_trace': bundle['standard_outcome'][1],
            'forced_OSD1': rank_followup_outcome(rank_followup_context, forced_osd1, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
            f'forced_OSD{full_order}': rank_followup_outcome(rank_followup_context, full_correction, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
            'best_snapshot_oracle_OSD1': best_correct,
            'dynamic_rank_OSD1': rank_followup_outcome(rank_followup_context, dynamic_osd1, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
            'oracle_rank_OSD1': rank_followup_outcome(rank_followup_context, oracle_osd1, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1],
            'Tesseract': int(rank_followup_case.tesseract_correct),
        }

        rank_followup_standard_correct = methods['standard_trace']
        for rank_followup_method, rank_followup_correct in methods.items():
            rank_followup_rows.append({
                'shot_id': rank_followup_case.shot_id,
                'distance': rank_followup_context.distance,
                'p': rank_followup_context.p,
                'split': followup_split(rank_followup_case.shot_id),
                'method': rank_followup_method,
                'logical_correct': int(rank_followup_correct),
                'logical_error': 1 - int(rank_followup_correct),
                'standard_trace_correct': rank_followup_standard_correct,
                'repair_vs_standard': int(not rank_followup_standard_correct and rank_followup_correct),
                'regression_vs_standard': int(rank_followup_standard_correct and (not rank_followup_correct)),
                'bp_syndrome_consistent': bundle['bp_outcome'][0],
                'endpoint_iteration': bundle['endpoint'],
                'selected_iteration': best_iteration if rank_followup_method == 'best_snapshot_oracle_OSD1' else bundle['endpoint'],
            })

    final_gap_trials = pd.DataFrame(rank_followup_rows)
    final_gap_trials.to_csv(final_gap_trial_path, index=False)

def paired_gap_bootstrap(wide, seed_offset):
    standard_error = wide['standard_trace'].to_numpy(dtype=float)
    tesseract_error = wide['Tesseract'].to_numpy(dtype=float)
    rng = np.random.default_rng(FOLLOWUP_CONFIG['random_seed'] + seed_offset)
    indices = rng.integers(0, len(wide), size=(FOLLOWUP_CONFIG['bootstrap_repetitions'], len(wide)))
    samples = standard_error[indices].mean(axis=1) - tesseract_error[indices].mean(axis=1)

    return (standard_error.mean() - tesseract_error.mean(), samples, indices)

rank_followup_gap_rows = []
if len(final_gap_trials):
    for (rank_followup_distance, rank_followup_p), rank_followup_frame in final_gap_trials.groupby(['distance', 'p'], sort=True):
        wide = rank_followup_frame.pivot(index='shot_id', columns='method', values='logical_error').sort_index()
        rank_followup_denominator, denominator_samples, indices = paired_gap_bootstrap(wide, len(rank_followup_gap_rows) + 7919 * int(rank_followup_distance))
        denominator_low, denominator_high = np.quantile(denominator_samples, [0.025, 0.975])
        resolved = denominator_low > 0
        rank_followup_gap_ids = wide[(wide.standard_trace == 1) & (wide.Tesseract == 0)].index
        standard_success_ids = wide[wide.standard_trace == 0].index

        for rank_followup_method in wide.columns:
            method_error = wide[rank_followup_method].to_numpy(dtype=float)
            rank_followup_ler = method_error.mean()
            rank_followup_eta = (wide.standard_trace.mean() - rank_followup_ler) / rank_followup_denominator if resolved else np.nan
            eta_samples = np.divide(wide.standard_trace.to_numpy(dtype=float)[indices].mean(axis=1) - method_error[indices].mean(axis=1), denominator_samples, out=np.full(len(denominator_samples), np.nan), where=denominator_samples > 0)
            finite = eta_samples[np.isfinite(eta_samples)]
            eta_low, eta_high = (float(np.quantile(finite, 0.025)), float(np.quantile(finite, 0.975))) if resolved and len(finite) else (np.nan, np.nan)
            rank_followup_gap_rows.append({
                'distance': rank_followup_distance,
                'p': rank_followup_p,
                'method': rank_followup_method,
                'shots': len(wide),
                'LER': rank_followup_ler,
                'standard_trace_LER': wide.standard_trace.mean(),
                'Tesseract_LER': wide.Tesseract.mean(),
                'gap_denominator': rank_followup_denominator,
                'gap_CI95_low': denominator_low,
                'gap_CI95_high': denominator_high,
                'gap_status': 'resolved' if resolved else 'unresolved',
                'gap_closure_fraction': rank_followup_eta,
                'closure_CI95_low': eta_low,
                'closure_CI95_high': eta_high,
                'repair_rate': 1 - wide.loc[rank_followup_gap_ids, rank_followup_method].mean() if len(rank_followup_gap_ids) else np.nan,
                'regression_rate': wide.loc[standard_success_ids, rank_followup_method].mean() if len(standard_success_ids) else np.nan,
            })

final_gap_regime_audit = pd.DataFrame(rank_followup_gap_rows, columns=['distance', 'p', 'method', 'shots', 'LER', 'standard_trace_LER', 'Tesseract_LER', 'gap_denominator', 'gap_CI95_low', 'gap_CI95_high', 'gap_status', 'gap_closure_fraction', 'closure_CI95_low', 'closure_CI95_high', 'repair_rate', 'regression_rate'])
final_gap_regime_audit.to_csv(rank_followup_data_paths.output('final_gap_regime_audit_csv'), index=False)
final_gap_regime_benchmark = final_gap_regime_audit[final_gap_regime_audit.gap_status == 'resolved'].reset_index(drop=True)

final_gap_regime_benchmark.to_csv(rank_followup_data_paths.output('final_gap_regime_benchmark_csv'), index=False)


## Experiment 10: selective OSD frontier

Vary selective OSD use and compare accuracy with invocation cost. A lower invocation fraction is useful only together with the paired decoding outcome.


In [ ]:
selective_feature_path = rank_followup_data_paths.output('selective_rank_features_csv')
if followup_stage_cached([selective_feature_path]):
    selective_rank_features = rank_followup_read_csv(selective_feature_path)
else:
    rank_followup_rows = []
    selective_parameter_lookup = {int(row.distance): tuple((float(getattr(row, f'beta_{name}')) for name in ['message', 'rank', 'conflict', 'sign'])) for row in rank_correction_parameters.itertuples(index=False)}
    for rank_followup_case in tqdm(rank_followup_unbiased_cases.itertuples(index=False), total=len(rank_followup_unbiased_cases), desc='selective OSD features', disable=FOLLOWUP_CONFIG['print_level'] == 1):
        rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace = followup_case_material(rank_followup_case)
        bundle = traced_decode_bundle(rank_followup_context, rank_followup_syndrome, rank_followup_truth, rank_followup_error, rank_followup_trace)
        rank_followup_endpoint = bundle['endpoint']
        features = prefix_features(rank_followup_context, rank_followup_trace, rank_followup_endpoint)
        rank_followup_k = min(10, rank_followup_context.H.shape[1])

        if rank_followup_endpoint > 1:
            previous_rank = rank_followup_ordinal_ranks(np.abs(rank_followup_trace.llr[rank_followup_endpoint - 1]))
            current_rank = features['ranks']
            left = set(np.flatnonzero(previous_rank <= rank_followup_k))
            right = set(np.flatnonzero(current_rank <= rank_followup_k))
            topk_stability = len(left & right) / len(left | right)
        else:
            topk_stability = 1.0

        corrected_score = corrected_rank_score(features, selective_parameter_lookup.get(rank_followup_context.distance, (0.0, 0.0, 0.0, 0.0)))
        osd1 = rank_followup_osd_from_reliability(rank_followup_context, rank_followup_syndrome, corrected_score, 1)
        osd1_correct = rank_followup_outcome(rank_followup_context, osd1, rank_followup_syndrome, rank_followup_truth, rank_followup_error)[1]
        rank_followup_rows.append({
            'shot_id': rank_followup_case.shot_id,
            'distance': rank_followup_context.distance,
            'p': rank_followup_context.p,
            'split': followup_split(rank_followup_case.shot_id),
            'bp_syndrome_consistent': bundle['bp_outcome'][0],
            'standard_correct': bundle['standard_outcome'][1],
            'osd1_correct': osd1_correct,
            'tesseract_correct': int(rank_followup_case.tesseract_correct),
            'benefit': int(not bundle['standard_outcome'][1] and osd1_correct),
            'rank_volatility': float(features['rank_volatility'].mean()),
            'topk_stability': topk_stability,
            'mean_conflict': float(features['extrinsic_conflict'].mean()),
            'minimum_confidence': float(features['current_abs_llr'].min()),
        })

    selective_rank_features = pd.DataFrame(rank_followup_rows)
    selective_rank_features.to_csv(selective_feature_path, index=False)

SELECTIVE_FEATURES = ['rank_volatility', 'topk_stability', 'mean_conflict', 'minimum_confidence']
frontier_rows = []
if len(selective_rank_features):
    for (rank_followup_distance, rank_followup_p), rank_followup_frame in selective_rank_features.groupby(['distance', 'p'], sort=True):
        rank_followup_development, test = (rank_followup_frame[rank_followup_frame.split == 'development'].copy(), rank_followup_frame[rank_followup_frame.split == 'test'].copy())
        risk_model = None
        if rank_followup_development.benefit.nunique() == 2:
            risk_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced'))
            risk_model.fit(rank_followup_development[SELECTIVE_FEATURES], rank_followup_development.benefit)
            rank_followup_development['predicted_misrank_risk'] = risk_model.predict_proba(rank_followup_development[SELECTIVE_FEATURES])[:, 1]
            test['predicted_misrank_risk'] = risk_model.predict_proba(test[SELECTIVE_FEATURES])[:, 1]
        else:
            rank_followup_development['predicted_misrank_risk'] = rank_followup_development.rank_volatility
            test['predicted_misrank_risk'] = test.rank_volatility

        standard_test_error = 1 - test.standard_correct.to_numpy(dtype=float)
        tesseract_test_error = 1 - test.tesseract_correct.to_numpy(dtype=float)
        rank_followup_denominator = standard_test_error.mean() - tesseract_test_error.mean() if len(test) else np.nan

        if len(test):
            rank_followup_rng = np.random.default_rng(FOLLOWUP_CONFIG['random_seed'] + 104729 * int(rank_followup_distance) + int(round(float(rank_followup_p) * 100000000.0)))
            indices = rank_followup_rng.integers(0, len(test), size=(FOLLOWUP_CONFIG['bootstrap_repetitions'], len(test)))
            denominator_samples = standard_test_error[indices].mean(axis=1) - tesseract_test_error[indices].mean(axis=1)
            gap_resolved = np.quantile(denominator_samples, 0.025) > 0
        else:
            gap_resolved = False

        rank_followup_bp_valid = test.bp_syndrome_consistent.astype(bool)
        policy_masks = {'standard_no_selective_osd': pd.Series(False, index=test.index)}
        for quantile in FOLLOWUP_CONFIG['trigger_quantiles']:
            volatility_threshold = rank_followup_development.rank_volatility.quantile(quantile)
            stability_threshold = rank_followup_development.topk_stability.quantile(1 - quantile)
            risk_threshold = rank_followup_development.predicted_misrank_risk.quantile(quantile)
            policy_masks[f'high_rank_volatility_q{quantile:g}'] = rank_followup_bp_valid & (test.rank_volatility >= volatility_threshold)
            policy_masks[f'low_topk_stability_q{quantile:g}'] = rank_followup_bp_valid & (test.topk_stability <= stability_threshold)
            policy_masks[f'high_misrank_risk_q{quantile:g}'] = rank_followup_bp_valid & (test.predicted_misrank_risk >= risk_threshold)
            policy_masks[f'volatility_or_stability_q{quantile:g}'] = rank_followup_bp_valid & ((test.rank_volatility >= volatility_threshold) | (test.topk_stability <= stability_threshold))

        for policy, trigger in policy_masks.items():
            selected_correct = np.where(trigger, test.osd1_correct, test.standard_correct)
            rank_followup_ler = 1 - float(np.mean(selected_correct)) if len(test) else np.nan
            standard_ler = 1 - test.standard_correct.mean() if len(test) else np.nan
            closure = (standard_ler - rank_followup_ler) / rank_followup_denominator if gap_resolved and rank_followup_denominator > 0 else np.nan
            gap_mask = (test.standard_correct == 0) & (test.tesseract_correct == 1)
            success_mask = test.standard_correct == 1
            frontier_rows.append({
                'distance': rank_followup_distance,
                'p': rank_followup_p,
                'policy': policy,
                'test_shots': len(test),
                'invocation_fraction': trigger.mean() if len(test) else np.nan,
                'valid_bp_invocation_fraction': trigger[test.bp_syndrome_consistent == 1].mean() if (test.bp_syndrome_consistent == 1).any() else np.nan,
                'LER': rank_followup_ler,
                'standard_trace_LER': standard_ler,
                'gap_closure_fraction': closure,
                'repair_rate': float(np.mean(selected_correct[gap_mask] == 1)) if gap_mask.any() else np.nan,
                'regression_rate': float(np.mean(selected_correct[success_mask] == 0)) if success_mask.any() else np.nan,
                'gap_status': 'resolved' if gap_resolved else 'unresolved',
                'risk_model_status': 'complete' if risk_model is not None else 'rank_volatility_fallback',
            })

selective_rank_osd_frontier = pd.DataFrame(frontier_rows, columns=['distance', 'p', 'policy', 'test_shots', 'invocation_fraction', 'valid_bp_invocation_fraction', 'LER', 'standard_trace_LER', 'gap_closure_fraction', 'repair_rate', 'regression_rate', 'gap_status', 'risk_model_status'])
selective_rank_osd_frontier.to_csv(rank_followup_data_paths.output('selective_rank_osd_frontier_csv'), index=False)


## Required outputs and figures

CSV files are persisted; figures are deliberately generated inline only.


In [ ]:
required_followup_stats = ['repair_topk_coverage.csv', 'repair_rank_vs_osd_success.csv', 'misranking_feature_ablation.csv', 'oracle_rank_ceiling.csv', 'snapshot_oracle_ceiling.csv', 'snapshot_rule_benchmark.csv', 'rank_correction_ablation.csv', 'pairwise_rank_model.csv', 'bp_parameter_rank_quality.csv', 'final_gap_regime_benchmark.csv', 'selective_rank_osd_frontier.csv']
required_followup_traces = ['repair_rank_trajectories.csv', 'misranked_variables.csv', 'oracle_rank_trials.csv', 'snapshot_oracle_trials.csv', 'rank_correction_trials.csv']
assert all((rank_followup_data_paths.output('name', name=name).is_file() for name in required_followup_stats))
assert all((rank_followup_data_paths.output('name_1', name=name).is_file() for name in required_followup_traces))

if len(repair_topk_coverage):
    sns.relplot(
        data=repair_topk_coverage[repair_topk_coverage.alignment == 'absolute'],
        x='aligned_iteration',
        y='mean_Q_K',
        hue='K',
        col='distance',
        kind='line',
    )
    plt.suptitle('Repair-variable Top-K coverage', y=1.03)
    plt.show()

if len(misranking_feature_ablation):
    plot_data = misranking_feature_ablation[misranking_feature_ablation.analysis_type == 'held_out_classifier']
    if len(plot_data):
        sns.catplot(
            data=plot_data,
            x='feature_set',
            y='AUROC',
            hue='K',
            col='distance',
            kind='bar',
        )
        plt.xticks(rotation=30, ha='right')
        plt.suptitle('Held-out mis-ranking predictor ablation', y=1.03)

        plt.show()

if len(oracle_rank_ceiling):
    oracle_plot = oracle_rank_ceiling[~oracle_rank_ceiling.variant.str.startswith('one_variable_')]
    sns.catplot(
        data=oracle_plot,
        x='variant',
        y='logical_correct',
        hue='osd_order',
        col='distance',
        kind='bar',
    )
    plt.xticks(rotation=35, ha='right')
    plt.suptitle('Oracle reliability-rank ceiling', y=1.03)

    plt.show()

if len(snapshot_oracle_ceiling):
    sns.catplot(
        data=snapshot_oracle_ceiling,
        x='rule',
        y='logical_correct',
        hue='osd_order',
        col='distance',
        kind='bar',
    )
    plt.xticks(rotation=30, ha='right')
    plt.suptitle('Snapshot-selection ceiling', y=1.03)

    plt.show()

if len(rank_correction_ablation):
    sns.catplot(
        data=rank_correction_ablation,
        x='method',
        y='logical_correct',
        hue='osd_order',
        col='distance',
        kind='bar',
    )
    plt.xticks(rotation=30, ha='right')
    plt.suptitle('Held-out dynamics-aware rank correction', y=1.03)

    plt.show()

resolved_gap_plot = final_gap_regime_benchmark[final_gap_regime_benchmark.gap_status == 'resolved']
if len(resolved_gap_plot):
    sns.relplot(
        data=resolved_gap_plot,
        x='p',
        y='gap_closure_fraction',
        hue='method',
        col='distance',
        kind='line',
        marker='o',
    )
    plt.axhline(0, color='black', linewidth=0.8)
    plt.suptitle('Resolved trace-native gap closure', y=1.03)

    plt.show()

followup_status
